## Implementation of GAN (Model-2)

In [ ]:
# Imports and setup
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# Config / hyperparameters
CSV_PATH = "../data/gan/train.csv"
FEATURE_PREFIX = "t_"
NUM_FEATURES = 100             # t_0 .. t_99
MALWARE_COL = "malware"        # column marking malware rows (1 = malware)
latent_dim = 64
generator_hidden = [256, 256]
discriminator_hidden = [256, 128]
batch_size = 128
epochs = 2000
save_dir = "saved_models"
learning_rate = 2e-4
beta1 = 0.5
latent_inversion_steps = 500
inversion_lr = 0.05
alpha = 1.0
beta = 1.0
seed = 42

np.random.seed(seed)
tf.random.set_seed(seed)
os.makedirs(save_dir, exist_ok=True)


In [2]:
def build_generator(latent_dim, out_dim, hidden_layers):
    inp = layers.Input(shape=(latent_dim,))
    x = inp
    for h in hidden_layers:
        x = layers.Dense(h)(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.BatchNormalization()(x)
    x = layers.Dense(out_dim, activation='linear')(x)
    return models.Model(inp, x, name="generator")


def build_discriminator(in_dim, hidden_layers):
    inp = layers.Input(shape=(in_dim,))
    x = inp
    for h in hidden_layers:
        x = layers.Dense(h)(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.3)(x)
    x = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, x, name="discriminator")


In [3]:
print("Loading CSV:", CSV_PATH)
df = pd.read_csv(CSV_PATH)

feature_cols = [f"t_{i}" for i in range(NUM_FEATURES)]
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected feature columns: {missing}")

if MALWARE_COL not in df.columns:
    raise ValueError(f"Missing malware column '{MALWARE_COL}'")

# Use only malware rows for training the GAN
malware_df = df[df[MALWARE_COL] == 1].copy()
if len(malware_df) == 0:
    raise ValueError("No malware rows found (malware == 1).")

X = malware_df[feature_cols].values.astype('float32')
print("Malware training rows:", X.shape[0])

# Standardize features
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

# Split
X_train, X_val = train_test_split(X_scaled, test_size=0.1, random_state=seed)

# Create TF dataset
dataset = tf.data.Dataset.from_tensor_slices(X_train.astype('float32')).shuffle(10000).batch(batch_size, drop_remainder=True)

Loading CSV: data/gan/train.csv


FileNotFoundError: [Errno 2] No such file or directory: 'data/gan/train.csv'

In [ ]:
generator = build_generator(latent_dim, NUM_FEATURES, generator_hidden)
discriminator = build_discriminator(NUM_FEATURES, discriminator_hidden)

gen_optimizer = optimizers.Adam(learning_rate, beta_1=beta1)
disc_optimizer = optimizers.Adam(learning_rate, beta_1=beta1)
bce = losses.BinaryCrossentropy(from_logits=False)

In [ ]:
@tf.function
def train_step(real_batch):
    batch_size_local = tf.shape(real_batch)[0]
    random_latent = tf.random.normal([batch_size_local, latent_dim])
    generated = generator(random_latent, training=True)

    real_labels = tf.ones((batch_size_local, 1))
    fake_labels = tf.zeros((batch_size_local, 1))

    # Train Discriminator
    with tf.GradientTape() as disc_tape:
        d_real = discriminator(real_batch, training=True)
        d_fake = discriminator(generated, training=True)
        d_loss_real = bce(real_labels, d_real)
        d_loss_fake = bce(fake_labels, d_fake)
        d_loss = d_loss_real + d_loss_fake
    grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(grads, discriminator.trainable_variables))

    # Train Generator
    random_latent = tf.random.normal([batch_size_local, latent_dim])
    with tf.GradientTape() as gen_tape:
        generated = generator(random_latent, training=True)
        d_fake_for_g = discriminator(generated, training=True)
        g_loss = bce(real_labels, d_fake_for_g)
    grads = gen_tape.gradient(g_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(grads, generator.trainable_variables))

    return d_loss, g_loss

In [ ]:
print("Starting training loop...")
d_losses, g_losses = [], []
for epoch in range(epochs):
    for real_batch in dataset:
        d_loss, g_loss = train_step(real_batch)
    if (epoch + 1) % 50 == 0 or epoch == 0:
        d_losses.append(float(d_loss.numpy()))
        g_losses.append(float(g_loss.numpy()))
        print(f"Epoch {epoch+1}/{epochs}  d_loss={d_losses[-1]:.4f}  g_loss={g_losses[-1]:.4f}")

In [ ]:
generator.save(os.path.join(save_dir, "generator.h5"))
discriminator.save(os.path.join(save_dir, "discriminator.h5"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.save"))
print("✅ Models and scaler saved to", save_dir)

In [ ]:
def disc_probability(samples_raw):
    samples_scaled = scaler.transform(np.asarray(samples_raw))
    probs = discriminator(samples_scaled.astype('float32'), training=False).numpy().reshape(-1)
    return probs


def invert_latent(x_raw, init_z=None, steps=latent_inversion_steps, lr=inversion_lr):
    x = scaler.transform(np.asarray(x_raw).reshape(1, -1)).astype('float32')
    if init_z is None:
        z = tf.Variable(tf.random.normal([1, latent_dim]), dtype=tf.float32)
    else:
        z = tf.Variable(init_z.reshape(1, latent_dim).astype('float32'))
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    for i in range(steps):
        with tf.GradientTape() as tape:
            generated = generator(z, training=False)
            loss = tf.reduce_mean(tf.square(generated - x))
        grads = tape.gradient(loss, [z])
        optimizer.apply_gradients(zip(grads, [z]))

    generated_final = generator(z, training=False).numpy().reshape(-1)
    generated_orig = scaler.inverse_transform(generated_final.reshape(1, -1)).reshape(-1)
    recon_error = np.linalg.norm(generated_orig - x_raw)
    return z.numpy().reshape(-1), generated_orig, float(recon_error)


def anomaly_score(x_raw, alpha=alpha, beta=beta, use_disc=True, use_recon=True, inversion_steps=None):
    if use_disc:
        disc_prob = float(disc_probability(np.atleast_2d(x_raw))[0])
    else:
        disc_prob = 0.0
    if use_recon:
        steps = inversion_steps if inversion_steps is not None else latent_inversion_steps
        _, _, recon_error = invert_latent(x_raw, steps=steps)
    else:
        recon_error = 0.0
    score = alpha * recon_error - beta * disc_prob
    return {
        "score": float(score),
        "recon_error": float(recon_error),
        "disc_prob": float(disc_prob)
    }

In [ ]:
if X_val.shape[0] > 0:
    sample_scaled = X_val[0]
    sample_orig = scaler.inverse_transform(sample_scaled.reshape(1, -1)).reshape(-1)
    print("Computing anomaly score on a validation sample...")
    res = anomaly_score(sample_orig, inversion_steps=200)
    print("Anomaly score (lower = more normal):", res)
else:
    print("No validation samples available.")